In [3]:
# 读取LMDB文件中的数据
import lmdb
import pickle

env = lmdb.open('/media/liud/Liud_FX2T/dataset/Cu_self/test_1', map_size=1099511627776, readonly=True, lock=False)

# 遍历所有键值对，并只打印前 5 个数据
print("First 5 entries in the database:")
count = 0
with env.begin(write=False) as txn:
    cursor = txn.cursor()
    for key, value in cursor:
        if value is not None:
            try:
                data = pickle.loads(value)
                print(f"Key: {key.decode()}")
                print("Data:", data)
                print("atomic_numbers:", type(data.atomic_numbers))
                print(data['features'])
                # print("y", type(data.y))
                # print("tags:", data.tags)
                # print("pos:", data.pos)
                # print(("distances:", len(data.distances)))
                # print("edge_index:", data.edge_index)
                # print("distances:", data.distances)
                print("features:", type(data.features))
            except Exception as e:
                print(f"Error processing key {key}: {e}")
        else:
            print(f"No data found for key: {key.decode()}")
        count += 1
        if count >= 20:
            break


First 5 entries in the database:
Key: 0
Data: Data(edge_index=[2, 30], y=-0.8134933700000033, pos=[12, 3], atomic_numbers=[12], natoms=12, tags=[12], fixed=[12], features=[12, 74], distances=[12], cell_offsets=[30, 3], cell=[1, 3, 3], sid=10386, fid=0)
atomic_numbers: <class 'torch.Tensor'>
tensor([[3.3333e-01, 6.6667e-01, 0.0000e+00, 6.6667e-01, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 1.3011e-05, 1.3798e-02, 4.4601e-01, 8.1303e-01, 2.2399e-01,
         1.1941e-01, 2.6968e-02, 1.3368e-01, 1.6530e-01, 1.4875e-01, 1.5173e-02,
         9.0423e-03, 7.2116e-02, 4.3328e-02, 1.6408e-03, 1.7796e-06, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 4.3631e-04, 1.6755e-03,
         5.1238e-03, 1.2802e-02, 2.7124e-02, 5.0628e-02, 8.4735e-02, 1.2565e-01,
         1.6144e-01, 1.7861e-01, 1.7358e-01, 1.5286e-01, 1.2309e-01, 8.7993e-02,
         5.3273e-02, 2.6274e-02, 1.0311e-02, 3.1843e-03, 7.9

In [31]:
# 计算lmdb文件中的数据条目数
import lmdb

def count_entries_in_lmdb(lmdb_path):
    env = lmdb.open(lmdb_path, readonly=True, lock=False)
    with env.begin() as txn:
        cursor = txn.cursor()
        count = sum(1 for _ in cursor)
    return count

if __name__ == "__main__":
    lmdb_path = "/home/zjy/code/mycode/ocp/test_data/Cu_self/test"  # 替换为你的LMDB文件路径
    num_entries = count_entries_in_lmdb(lmdb_path)
    print(f"Number of entries in the LMDB file: {num_entries}")


Number of entries in the LMDB file: 519


In [9]:
# # 读取LMDB文件中的数据，并查看结构
# import lmdb
# import pickle
# import numpy as np
# from ase import Atoms
# from ase.visualize import view
# 
# env = lmdb.open('/home/zjy/code/mycode/ocp/test_data/all_change/test', map_size=1099511627776, readonly=True, lock=False)
# 
# # 遍历所有键值对，并只打印第一个数据
# print("First entry in the database:")
# with env.begin(write=False) as txn:
#     cursor = txn.cursor()
#     for key, value in cursor:
#         if value is not None:
#             try:
#                 data = pickle.loads(value)
#                 print(f"Key: {key.decode()}")
#                 print("Data:", data)
#                 print("tags:", data.tags)
# 
#                 # 创建 Atoms 对象
#                 cell = np.array(data.cell).reshape((3, 3))
#                 atoms = Atoms(numbers=data.atomic_numbers, positions=data.pos, cell=cell, pbc=True)
# 
#                 # 使用 ase.view 显示结构
#                 view(atoms)
#                 break  # 只显示第一个数据
#             except Exception as e:
#                 print(f"Error processing key {key}: {e}")
#         else:
#             print(f"No data found for key: {key.decode()}")

# 读取LMDB文件中的数据，并查看结构
import lmdb
import pickle
import numpy as np
from ase import Atoms
from ase.visualize import view

def find_and_view_structure(lmdb_path, target_sid, target_fid):
    env = lmdb.open(lmdb_path, map_size=1099511627776, readonly=True, lock=False)

    # 遍历所有键值对，并查找目标 sid 和 fid 的数据
    with env.begin(write=False) as txn:
        cursor = txn.cursor()
        for key, value in cursor:
            if value is not None:
                try:
                    data = pickle.loads(value)
                    if data.sid == target_sid and data.fid == target_fid:
                        print(f"Found entry with sid={target_sid} and fid={target_fid}")
                        print(f"Key: {key.decode()}")
                        print("Data:", data)
                        print("tags:", data.tags)

                        # 创建 Atoms 对象
                        cell = np.array(data.cell).reshape((3, 3))
                        atoms = Atoms(numbers=data.atomic_numbers, positions=data.pos, cell=cell, pbc=True)

                        # 使用 ase.view 显示结构
                        view(atoms)
                        return  # 只显示第一个匹配的数据
                except Exception as e:
                    print(f"Error processing key {key}: {e}")
            else:
                print(f"No data found for key: {key.decode()}")

    print(f"No entry found with sid={target_sid} and fid={target_fid}")

# 使用示例
lmdb_path = '/home/zjy/code/mycode/Cu_data/data_analyse/test_end'
target_sid = 1320845  # 需要查找的sid
target_fid = 6       # 需要查找的fid

find_and_view_structure(lmdb_path, target_sid, target_fid)


Found entry with sid=1320845 and fid=6
Key: 10
Data: Data(edge_index=[2, 34], y=-1.6260464300000308, pos=[30, 3], cell=[1, 3, 3], atomic_numbers=[30], natoms=30, tags=[30], cell_offsets=[34, 3], fixed=[30], sid=1320845, fid=6, features=[30, 74], distances=[30])
tags: tensor([1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 2, 2, 0, 1, 1, 2, 2, 2, 1, 2, 1,
        1, 1, 0, 0, 1, 0])


In [21]:
# 筛选lmdb文件中的数据
import os
import lmdb
import pickle
from tqdm import tqdm

def is_data_format_correct(data):
    try:
        if (
            data.edge_index.shape[0] == 2 and
            data.pos.shape[1] == 3 and
            data.cell.shape == (1, 3, 3)
        ):
            return True
    except AttributeError:
        return False
    return False

def filter_and_rekey_data(input_lmdb_path, output_lmdb_path):
    env = lmdb.open(input_lmdb_path, readonly=True, lock=False)
    filtered_env = lmdb.open(output_lmdb_path, map_size=1099511627776)

    with env.begin(write=False) as txn, filtered_env.begin(write=True) as filtered_txn:
        cursor = txn.cursor()
        new_key = 0
        for key, value in tqdm(cursor):
            data = pickle.loads(value)
            if data.natoms >= 6 and is_data_format_correct(data):
                new_key_bytes = f"{new_key}".encode("ascii")
                filtered_txn.put(new_key_bytes, pickle.dumps(data))
                new_key += 1

    env.close()
    filtered_env.close()
    print(f"Filtered and rekeyed data written to {output_lmdb_path}")

if __name__ == "__main__":
    input_lmdb_path = '/home/zjy/code/mycode/ocp/test_data/all_oc20/val_4'  # 替换为你的输入 LMDB 文件路径
    output_lmdb_path = '/home/zjy/code/mycode/ocp/test_data/all_oc20/val_5'  # 替换为你的输出 LMDB 文件路径
    filter_and_rekey_data(input_lmdb_path, output_lmdb_path)



415it [00:00, 1080.04it/s]


Filtered and rekeyed data written to /home/zjy/code/mycode/ocp/test_data/all_oc20/val_5


In [17]:
from tqdm import tqdm
import lmdb
import pickle
import numpy as np
import torch
from torch_geometric.data import Data
from ase import Atoms
from ase.visualize import view

def update_mdb_data(old_lmdb_path, new_lmdb_path, num_entries=10):
    # 打开源LMDB文件
    old_env = lmdb.open(old_lmdb_path, readonly=True, lock=False, map_size=int(1e12))
    # 打开新的LMDB文件
    new_env = lmdb.open(new_lmdb_path, map_size=int(1e12))

    with old_env.begin(write=False) as old_txn, new_env.begin(write=True) as new_txn:
        cursor = old_txn.cursor()

        # 使用tqdm显示进度条
        for i, (key, value) in enumerate(tqdm(cursor, total=num_entries, desc="Processing")):
            if i >= num_entries:
                break
            data = pickle.loads(value)

            # 更新数据，删除tags==2的原子相关数据
            mask = data.tags != 2
            pos = data.pos[mask]
            atomic_numbers = data.atomic_numbers[mask]
            natoms = int(mask.sum())
            tags = data.tags[mask]
            fixed = data.fixed[mask]
            features = data.features[mask]
            distances = data.distances[mask]

            # 筛选 edge_index
            old_to_new_idx = -torch.ones(data.tags.size(0), dtype=torch.long)
            old_to_new_idx[mask] = torch.arange(natoms)

            if hasattr(data, 'edge_index') and data.edge_index is not None and len(data.edge_index.shape) > 1 and data.edge_index.shape[1] > 0:
                new_edge_index = []
                new_cell_offsets = []
                for j, edge in enumerate(data.edge_index.T):
                    if mask[edge[0]] and mask[edge[1]]:
                        new_edge_index.append([old_to_new_idx[edge[0]].item(), old_to_new_idx[edge[1]].item()])
                        new_cell_offsets.append(data.cell_offsets[j].tolist())
                edge_index = torch.tensor(new_edge_index, dtype=torch.long).T if new_edge_index else torch.empty((2, 0), dtype=torch.long)
                cell_offsets = torch.tensor(new_cell_offsets, dtype=torch.float) if new_cell_offsets else torch.empty((0, 3), dtype=torch.float)
            else:
                edge_index = torch.empty((2, 0), dtype=torch.long)
                cell_offsets = torch.empty((0, 3), dtype=torch.float)

            # 检查并确保 cell 属性存在
            cell = data.cell if hasattr(data, 'cell') else torch.empty((1, 3, 3), dtype=torch.float)

            # 将更新后的数据写入新的LMDB文件
            new_data = Data(
                pos=pos,
                atomic_numbers=atomic_numbers,
                natoms=natoms,
                tags=tags,
                fixed=fixed,
                features=features,
                distances=distances,
                edge_index=edge_index,
                cell_offsets=cell_offsets,
                cell=cell,  # 添加 cell 属性
                y=data.y,
                sid=data.sid,
                fid=data.fid
            )
            new_value = pickle.dumps(new_data)
            new_txn.put(key, new_value)

    old_env.close()
    new_env.close()

# 使用示例
old_lmdb_path = '/home/zjy/code/mycode/ocp/test_data/all_oc20/test_3'
new_lmdb_path = '/home/zjy/code/mycode/ocp/test_data/all_oc20/test_4'
update_mdb_data(old_lmdb_path, new_lmdb_path, num_entries=50000)

Processing:   1%|          | 421/50000 [00:01<02:16, 363.02it/s]


In [6]:
# 从LMDB文件中复制随机条目
import lmdb
import pickle
import random

def copy_random_entries(old_lmdb_path, new_lmdb_path, num_entries):
    old_env = lmdb.open(old_lmdb_path, readonly=True, lock=False)
    new_env = lmdb.open(new_lmdb_path, map_size=int(1e12))

    all_keys = []
    with old_env.begin(write=False) as txn:
        cursor = txn.cursor()
        for key, _ in cursor:
            all_keys.append(key)
    
    # 随机选择 num_entries 个键
    selected_keys = random.sample(all_keys, min(num_entries, len(all_keys)))

    with old_env.begin(write=False) as old_txn, new_env.begin(write=True) as new_txn:
        for new_key_int, old_key in enumerate(selected_keys):
            value = old_txn.get(old_key)
            if value is not None:
                try:
                    data = pickle.loads(value)
                    new_key = f"{new_key_int}".encode('ascii')
                    new_txn.put(new_key, pickle.dumps(data, protocol=-1))
                except Exception as e:
                    print(f"Error processing key {old_key.decode()}: {e}")
    
    new_env.sync()
    new_env.close()
    old_env.close()

# 使用示例
old_lmdb_path = '/home/zjy/code/mycode/ocp/test_data/all_CNN/test/test_050_070'
new_lmdb_path = '/home/zjy/code/mycode/ocp/test_data/all_CNN/test/test_050_070_1'
copy_random_entries(old_lmdb_path, new_lmdb_path, num_entries=50)


In [7]:
# 合并两个LMDB文件
import lmdb
import pickle
import random

def open_lmdb_files(lmdb_path1, lmdb_path2, merged_lmdb_path):
    print(f"Opening LMDB files: {lmdb_path1}, {lmdb_path2}")
    env1 = lmdb.open(lmdb_path1, readonly=True, lock=False)
    env2 = lmdb.open(lmdb_path2, readonly=True, lock=False)
    merged_env = lmdb.open(merged_lmdb_path, map_size=int(1e12))
    print("Successfully opened all LMDB files.")
    return env1, env2, merged_env

def merge_lmdb_files(env1, env2, merged_env, num_samples=893):
    try:
        with env1.begin(write=False) as txn1, env2.begin(write=False) as txn2:
            print("Begin transactions on env1 and env2")
            cursor1 = list(txn1.cursor())
            cursor2 = list(txn2.cursor())
            
            # Randomly select samples from each cursor
            combined_cursor = random.sample(cursor1 + cursor2, num_samples)
            
            new_key_index = 0
            with merged_env.begin(write=True) as merged_txn:
                print("Begin transaction on merged_env")
                
                # Merge entries from combined_cursor
                for key, value in combined_cursor:
                    new_key = f"{new_key_index}".encode()
                    merged_txn.put(new_key, value)
                    new_key_index += 1
                
                print(f"Total entries merged: {new_key_index}")
                merged_txn.commit()
        
        print("Closing LMDB environments")
        env1.close()
        env2.close()
        merged_env.close()
        print(f"Merged LMDB created at {merged_lmdb_path} with {new_key_index} entries.")
    
    except Exception as e:
        print(f"Error: {e}")
        if 'env1' in locals() and env1 is not None:
            env1.close()
        if 'env2' in locals() and env2 is not None:
            env2.close()
        if 'merged_env' in locals() and merged_env is not None:
            merged_env.close()

# 示例路径，替换为实际路径
lmdb_path1 = '/home/zjy/code/mycode/ocp/test_data/all_CNN/test/test_050'
lmdb_path2 = '/home/zjy/code/mycode/ocp/test_data/all_CNN/test/test_050_070_1'
merged_lmdb_path = '/home/zjy/code/mycode/ocp/test_data/all_CNN/test/test'

env1, env2, merged_env = open_lmdb_files(lmdb_path1, lmdb_path2, merged_lmdb_path)
merge_lmdb_files(env1, env2, merged_env)


Opening LMDB files: /home/zjy/code/mycode/ocp/test_data/all_CNN/test/test_050, /home/zjy/code/mycode/ocp/test_data/all_CNN/test/test_050_070_1
Successfully opened all LMDB files.
Begin transactions on env1 and env2
Begin transaction on merged_env
Total entries merged: 893
Error: Attempt to operate on closed/deleted/dropped object.


In [36]:
# 预测数据的csv储存
import numpy as np
import lmdb
import pickle
import pandas as pd

# 读取 npz 文件中的数据
data = np.load('/home/zjy/code/mycode/ocp/results/2024-06-25-17-38-08/is2re_predictions.npz')

# 提取 ids1, ids2 和 energy
ids1 = data['ids1']
ids2 = data['ids2']
energy = data['energy']

# 将 ids1 和 ids2 转换为数值
ids1_numeric = ids1.astype(int)
ids2_numeric = ids2.astype(int)

# 创建一个索引列表，先根据 ids1_numeric 排序，然后再根据 ids2_numeric 排序
sorted_indices = np.lexsort((ids2_numeric, ids1_numeric))

# 根据排序后的索引获取排序后的 ids1, ids2 和 energy 列表
ids1_sorted = ids1_numeric[sorted_indices]
ids2_sorted = ids2_numeric[sorted_indices]
energy_sorted = energy[sorted_indices]

# 打开 LMDB 文件
env = lmdb.open('/home/zjy/code/mycode/ocp/test_data/test_10000', readonly=True)

# 列表用于存储 (sid, fid, y) 元组
data_list = []

# 从 LMDB 文件中读取数据并提取 sid, fid 和 y 值
with env.begin(write=False) as txn:
    cursor = txn.cursor()
    for key, value in cursor:
        info = pickle.loads(value)
        data_list.append((info.sid, info.fid, info.y))

# 关闭 LMDB 文件
env.close()

# 将数据转换为 numpy 数组以便于排序
data_array = np.array(data_list, dtype=[('sid', 'i4'), ('fid', 'i4'), ('y', 'f8')])

# 按照 sid 和 fid 对 data_array 进行排序
sorted_data_array = np.sort(data_array, order=['sid', 'fid'])

# 提取排序后的 sid、fid 和 y 值列表
sid_sorted = sorted_data_array['sid']
fid_sorted = sorted_data_array['fid']
truth_sorted = sorted_data_array['y']

# 创建 DataFrame
df = pd.DataFrame({
    'sid': sid_sorted,
    'fid': fid_sorted,
    'truth': truth_sorted,
    'predict': energy_sorted
})

# 保存为 CSV 文件
csv_path = '/home/zjy/code/mycode/ocp/test_data/test.csv'
df.to_csv(csv_path, index=False)

print(f"CSV file created at {csv_path}")


CSV file created at /home/zjy/code/mycode/ocp/test_data/test.csv


In [40]:
# key值的重新赋予
import lmdb
import pickle

def reassign_keys_in_lmdb(input_lmdb_path, output_lmdb_path):
    # 打开旧的LMDB文件
    env_old = lmdb.open(input_lmdb_path, readonly=True, lock=False)
    
    # 打开新的LMDB文件
    env_new = lmdb.open(output_lmdb_path, map_size=int(1e12))
    
    with env_old.begin(write=False) as txn_old, env_new.begin(write=True) as txn_new:
        cursor = txn_old.cursor()
        
        new_key_index = 0
        for old_key, value in cursor:
            try:
                # 将值解码
                data = pickle.loads(value)
                
                # 生成新的key
                new_key = f"{new_key_index}".encode("ascii")
                
                # 将新的键值对放入新LMDB
                txn_new.put(new_key, pickle.dumps(data, protocol=-1))
                
                new_key_index += 1
            except Exception as e:
                print(f"Error processing key {old_key.decode()}: {e}")
        
        txn_new.commit()
        env_new.sync()
    
    env_old.close()
    env_new.close()

# 使用示例
input_lmdb_path = '/home/zjy/code/mycode/ocp/test_data/Cu_self/train+val+test_0'
output_lmdb_path = '/home/zjy/code/mycode/ocp/test_data/Cu_self/train+val+test_1'
reassign_keys_in_lmdb(input_lmdb_path, output_lmdb_path)


Error: Attempt to operate on closed/deleted/dropped object.

In [43]:
import lmdb
import pickle
import numpy as np
import torch

def convert_distances_to_tensor(old_lmdb_path, new_lmdb_path):
    # 打开旧的 LMDB 文件
    old_env = lmdb.open(old_lmdb_path, map_size=int(1e12), readonly=True)
    # 打开新的 LMDB 文件
    new_env = lmdb.open(new_lmdb_path, map_size=int(1e12))
    
    with old_env.begin(write=False) as old_txn, new_env.begin(write=True) as new_txn:
        cursor = old_txn.cursor()
        
        for key, value in cursor:
            data = pickle.loads(value)
            
            try:
                # 检查是否有 distances 属性，并且它是一个 numpy 数组
                if hasattr(data, 'distances') and isinstance(data.distances, np.ndarray):
                    # 将 distances 从 numpy 数组转换为 PyTorch Tensor
                    data.distances = torch.tensor(data.distances)
                    
                    # 将修改后的数据存储到新的 LMDB 文件中
                    new_txn.put(key, pickle.dumps(data))
                else:
                    # 如果没有 distances 属性或它不是 numpy 数组，直接存储原始数据
                    new_txn.put(key, value)
                    
            except Exception as e:
                print(f"Error processing key {key.decode()}: {e}")

# 使用示例
old_lmdb_path = '/home/zjy/code/mycode/Cu_data/test_create_1'
new_lmdb_path = '/mycode/Cu_data/test_create_end'
convert_distances_to_tensor(old_lmdb_path, new_lmdb_path)


In [59]:
import lmdb
import pickle
from tqdm import tqdm

def extract_sid_tags(lmdb_path):
    sid_tags_dict = {}
    env = lmdb.open(lmdb_path, readonly=True, lock=False, map_size=int(1e12))
    with env.begin(write=False) as txn:
        cursor = txn.cursor()
        for key, value in cursor:
            data = pickle.loads(value)
            sid_tags_dict[data.sid] = data.tags
    env.close()
    return sid_tags_dict

def update_tags(lmdb_path, sid_tags_dict, new_lmdb_path):
    old_env = lmdb.open(lmdb_path, readonly=True, lock=False, map_size=int(1e12))
    new_env = lmdb.open(new_lmdb_path, map_size=int(1e12))

    with old_env.begin(write=False) as old_txn, new_env.begin(write=True) as new_txn:
        cursor = old_txn.cursor()

        for key, value in tqdm(cursor, desc="Updating tags"):
            data = pickle.loads(value)
            if data.sid in sid_tags_dict:
                data.tags = sid_tags_dict[data.sid]
            new_txn.put(key, pickle.dumps(data))
    
    old_env.close()
    new_env.close()

# 提取第一个LMDB文件中的sid和tags
lmdb_path1 = '/home/zjy/code/mycode/Cu_data/test_end_l'
sid_tags_dict = extract_sid_tags(lmdb_path1)

# 更新第二个LMDB文件中的tags
lmdb_path2 = '/home/zjy/code/mycode/Cu_data/test_create_end'
new_lmdb_path = '/home/zjy/code/mycode/Cu_data/test_end_updated'
update_tags(lmdb_path2, sid_tags_dict, new_lmdb_path)


Updating tags: 8732it [00:07, 1110.02it/s]


In [28]:
import lmdb
import random
import os

def shuffle_lmdb(src_path, dest_path):
    # 打开源LMDB文件
    env = lmdb.open(src_path, readonly=True)
    entries = env.stat()['entries']

    # 随机选择键
    with env.begin() as txn:
        keys = list(txn.cursor().iternext(keys=True, values=False))
    random.shuffle(keys)

    # 定义一个函数来保存数据到新的LMDB文件，并重新排序键值
    def save_to_new_lmdb(keys, dest_path):
        if not os.path.exists(dest_path):
            os.makedirs(dest_path)
        dest_env = lmdb.open(dest_path, map_size=1099511627776)
        with env.begin() as src_txn, dest_env.begin(write=True) as dest_txn:
            for idx, key in enumerate(keys):
                value = src_txn.get(key)
                dest_txn.put(str(idx).encode(), value)
        dest_env.close()

    # 保存数据到新的LMDB文件
    save_to_new_lmdb(keys, dest_path)

    # 关闭源LMDB文件
    env.close()

# 使用示例
src_path = "/home/zjy/code/mycode/ocp/test_data/middle/train+val_end"
dest_path = "/home/zjy/code/mycode/ocp/test_data/middle/train+val_end_split"
shuffle_lmdb(src_path, dest_path)


In [11]:
import lmdb
import pickle
import torch

def extract_features(lmdb_path):
    # 打开LMDB文件
    env = lmdb.open(lmdb_path, readonly=True, lock=False)
    
    features_dict = {}
    found_counts = {1: False, 2: False, 3: False, 4: False, 5: False, 6: False}

    with env.begin(write=False) as txn:
        cursor = txn.cursor()
        for key, value in cursor:
            data = pickle.loads(value)
            tags = data.tags
            count_tags_2 = torch.sum(tags == 2).item()
            
            if count_tags_2 in found_counts and not found_counts[count_tags_2]:
                features_dict[count_tags_2] = data.features[0]
                found_counts[count_tags_2] = True
            
            if all(found_counts.values()):
                break

    env.close()
    return features_dict

# 示例路径，替换为实际路径
lmdb_path = '/home/zjy/code/mycode/ocp/test_data/all_oc20/train_3'
features_dict = extract_features(lmdb_path)

print(features_dict)


{1: tensor([1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0.]), 2: tensor([7.0711e-01, 0.0000e+00, 0.0000e+00, 7.0711e-01, 0.0000e+00, 0.0000e+00,
        3.4940e-06, 8.8449e-03, 4.6696e-01, 8.8126e-01, 7.2542e-02, 1.5078e-04,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e

In [14]:
import lmdb
import pickle
import torch
from tqdm import tqdm

def replace_features(lmdb_path, features_dict):
    # 打开LMDB文件
    env = lmdb.open(lmdb_path, map_size=int(1e12))
    
    with env.begin(write=True) as txn:
        cursor = txn.cursor()
        for key, value in tqdm(cursor):
            data = pickle.loads(value)
            tags = data.tags
            count_tags_2 = torch.sum(tags == 2).item()
            
            if count_tags_2 in features_dict:
                new_feature = features_dict[count_tags_2]
                rows = data.features.shape[0]
                data.features = torch.stack([new_feature] * rows, dim=0)
                
                new_value = pickle.dumps(data)
                txn.put(key, new_value)

    env.close()

# 示例路径，替换为实际路径
lmdb_path = '/home/zjy/code/mycode/ocp/test_data/all_oc20/test_3'

# features_dict = {3: torch.tensor([5.7735e-01, 5.7735e-01, 0.0000e+00,5.7735e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 5.1977e-05, 3.3806e-02, 5.6375e-01, 7.6662e-01, 2.3320e-01, 4.7989e-03, 4.9368e-02, 1.8819e-01, 3.2729e-02, 1.6371e-04, 1.7055e-08, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 7.1596e-08, 1.0739e-06, 1.1742e-05, 9.9124e-05, 6.4773e-04, 3.2692e-03, 1.2750e-02, 3.8428e-02, 8.9526e-02, 1.6123e-01, 2.2449e-01, 2.4166e-01, 2.0113e-01, 1.2942e-01, 6.4379e-02, 2.4756e-02, 7.3575e-03, 1.6899e-03, 2.9980e-04, 4.1049e-05, 4.3786e-06, 4.1049e-07, 0.0000e+00, 0.0000e+00, 0.0000e+00, 1.4319e-07, 2.0405e-06, 2.1300e-05, 1.7437e-04, 1.1092e-03, 5.4945e-03, 2.1237e-02, 6.4158e-02, 1.5178e-01, 2.8162e-01, 4.1040e-01, 4.7014e-01, 4.2350e-01, 2.9991e-01, 1.6683e-01]), 1: torch.tensor([1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]), 2: torch.tensor([1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 1.2108e-04, 6.4877e-02, 8.6463e-01, 4.9810e-01, 1.0423e-02, 4.5412e-06, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00])}
features_dict = {1: torch.tensor([1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0.]), 2: torch.tensor([7.0711e-01, 0.0000e+00, 0.0000e+00, 7.0711e-01, 0.0000e+00, 0.0000e+00,
        3.4940e-06, 8.8449e-03, 4.6696e-01, 8.8126e-01, 7.2542e-02, 1.5078e-04,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00]), 3: torch.tensor([5.7735e-01, 5.7735e-01, 0.0000e+00, 5.7735e-01, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 1.3068e-06, 4.4517e-03, 3.3504e-01, 9.1983e-01, 1.2302e-01,
        5.5582e-04, 1.7898e-02, 1.5130e-01, 5.7399e-02, 7.2905e-04, 1.8046e-07,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 2.9062e-06, 2.8743e-05,
        2.1864e-04, 1.2851e-03, 5.8350e-03, 2.0474e-02, 5.5522e-02, 1.1639e-01,
        1.8862e-01, 2.3632e-01, 2.2892e-01, 1.7144e-01, 9.9269e-02, 4.4434e-02,
        1.5374e-02, 4.1108e-03, 8.4932e-04, 1.3563e-04, 1.6799e-05, 1.5147e-06,
        1.3770e-07, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        3.5442e-07, 4.4657e-06, 4.5153e-05, 3.5300e-04, 2.1318e-03, 9.9521e-03,
        3.5917e-02, 1.0023e-01, 2.1629e-01, 3.6100e-01, 4.6604e-01, 4.6537e-01,
        3.5944e-01, 2.1474e-01]), 5: torch.tensor([3.3333e-01, 6.6667e-01, 0.0000e+00, 6.6667e-01, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 4.9552e-05, 2.5735e-02, 3.5542e-01, 7.0864e-01, 5.3558e-01,
        2.3587e-02, 1.9636e-02, 1.7120e-01, 1.8701e-01, 1.1900e-01, 2.5355e-02,
        4.6257e-02, 2.5589e-02, 1.0502e-03, 1.9918e-02, 2.7903e-02, 1.6458e-03,
        2.3298e-06, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 2.3075e-02, 2.8360e-02,
        3.5008e-02, 4.6206e-02, 6.0654e-02, 7.3274e-02, 8.2421e-02, 9.2924e-02,
        1.0928e-01, 1.2794e-01, 1.3855e-01, 1.3268e-01, 1.1132e-01, 8.4027e-02,
        6.0417e-02, 4.3509e-02, 3.1714e-02, 2.3820e-02, 1.9447e-02, 1.6919e-02,
        1.3870e-02, 9.6968e-03, 5.9598e-03, 4.7873e-03, 7.8197e-03, 1.6065e-02,
        2.9656e-02, 4.7813e-02, 6.8979e-02, 8.9675e-02, 1.0424e-01, 1.1075e-01,
        1.1984e-01, 1.5334e-01, 2.2748e-01, 3.3207e-01, 4.2522e-01, 4.5637e-01,
        4.0422e-01, 2.9385e-01]), 6: torch.tensor([5.7735e-01, 5.7735e-01, 0.0000e+00, 5.7735e-01, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 2.8257e-05, 2.2729e-02, 4.6499e-01, 6.7556e-01, 3.3809e-01,
        1.7472e-01, 1.3194e-01, 2.7868e-01, 2.3390e-01, 1.6730e-01, 1.8091e-02,
        4.6729e-03, 4.2708e-02, 4.5522e-02, 7.0598e-03, 4.3120e-05, 3.8583e-09,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 1.7603e-05, 1.0387e-04,
        4.7864e-04, 1.7317e-03, 4.9647e-03, 1.1468e-02, 2.1964e-02, 3.6398e-02,
        5.4773e-02, 7.7261e-02, 1.0232e-01, 1.2526e-01, 1.3942e-01, 1.3878e-01,
        1.2114e-01, 9.1753e-02, 6.2451e-02, 4.3624e-02, 3.6804e-02, 3.6264e-02,
        3.6499e-02, 3.6806e-02, 3.9823e-02, 4.7876e-02, 6.1751e-02, 8.1352e-02,
        1.0454e-01, 1.2556e-01, 1.3849e-01, 1.4387e-01, 1.4999e-01, 1.6817e-01,
        2.0816e-01, 2.7261e-01, 3.4795e-01, 4.0219e-01, 4.0251e-01, 3.4113e-01,
        2.4244e-01, 1.4377e-01]), 4: torch.tensor([7.0711e-01, 0.0000e+00, 0.0000e+00, 7.0711e-01, 0.0000e+00, 0.0000e+00,
        2.8752e-06, 7.7779e-03, 4.3478e-01, 8.6079e-01, 7.4694e-02, 1.6499e-04,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 1.5510e-07, 5.4020e-04,
        4.0079e-02, 1.4980e-01, 1.3556e-01, 1.4456e-01, 1.5721e-02, 5.0117e-05,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 7.3885e-08,
        2.0949e-04, 1.2261e-02, 2.5227e-02, 2.2833e-03, 1.9822e-04, 9.0530e-04,
        3.1952e-03, 8.7161e-03, 1.8382e-02, 2.9999e-02, 3.8036e-02, 3.8248e-02,
        3.3635e-02, 3.5026e-02, 5.4686e-02, 9.6370e-02, 1.4769e-01, 1.8514e-01,
        1.9467e-01, 1.8590e-01, 1.7711e-01, 1.6957e-01, 1.4887e-01, 1.0944e-01,
        6.4433e-02, 3.1039e-02, 1.7105e-02, 2.3545e-02, 5.0527e-02, 9.4680e-02,
        1.4327e-01, 1.7987e-01, 1.9791e-01, 2.0079e-01, 1.8886e-01, 1.5932e-01,
        1.2141e-01, 9.9479e-02, 1.1701e-01, 1.8128e-01, 2.7659e-01, 3.6300e-01,
        3.9130e-01, 3.3889e-01])}
replace_features(lmdb_path, features_dict)

print("Features replaced successfully.")


421it [00:00, 1023.95it/s]

Features replaced successfully.


In [36]:
# import lmdb
# import pickle
# from tqdm import tqdm
# 
# def reshape_features(input_lmdb_path, output_lmdb_path):
#     env = lmdb.open(input_lmdb_path, readonly=True, lock=False)
#     filtered_env = lmdb.open(output_lmdb_path, map_size=1099511627776)
# 
#     with env.begin(write=False) as txn, filtered_env.begin(write=True) as filtered_txn:
#         cursor = txn.cursor()
#         for key, value in tqdm(cursor):
#             data = pickle.loads(value)
# 
#             # 如果 features 维度为 [natoms, 74]，则将其转换为 [1, 74]
#             if data.features.shape[0] > 1 and data.features.shape[1] == 74:
#                 data.features = data.features[:1, :]  # 只保留第一个原子的特征向量
# 
#             # 将更新后的数据写入新的 LMDB 文件
#             new_value = pickle.dumps(data)
#             filtered_txn.put(key, new_value)
# 
#     env.close()
#     filtered_env.close()
#     print(f"Reshaped features and saved data to {output_lmdb_path}")
# 
# if __name__ == "__main__":
#     input_lmdb_path = '/home/zjy/code/mycode/ocp/test_data/all_oc20/train_5'  # 替换为你的输入 LMDB 文件路径
#     output_lmdb_path = '/home/zjy/code/mycode/ocp/test_data/all_oc20/train_6'  # 替换为你的输出 LMDB 文件路径
#     reshape_features(input_lmdb_path, output_lmdb_path)

import lmdb
import pickle
from tqdm import tqdm

def reshape_features(input_lmdb_path, output_lmdb_path):
    env = lmdb.open(input_lmdb_path, readonly=True, lock=False)
    filtered_env = lmdb.open(output_lmdb_path, map_size=1099511627776)

    with env.begin(write=False) as txn, filtered_env.begin(write=True) as filtered_txn:
        cursor = txn.cursor()
        for key, value in tqdm(cursor):
            data = pickle.loads(value)

            # 如果 features 维度为 [1, 74]，则将其转换为 [natoms, 74]
            if data.features.shape[0] == 1 and data.features.shape[1] == 74:
                # 假设 'natoms' 是数据中的一个字段，表示原子的数量
                natoms = data.natoms  
                data.features = data.features.repeat(natoms, 1)  # 复制到 [natoms, 74]

            # 将更新后的数据写入新的 LMDB 文件
            new_value = pickle.dumps(data)
            filtered_txn.put(key, new_value)

    env.close()
    filtered_env.close()
    print(f"Reshaped features and saved data to {output_lmdb_path}")

if __name__ == "__main__":
    input_lmdb_path = '/home/zjy/code/mycode/ocp/test_data/Cu_self/test_0'  # 替换为你的输入 LMDB 文件路径
    output_lmdb_path = '/home/zjy/code/mycode/ocp/test_data/Cu_self/test_1'  # 替换为你的输出 LMDB 文件路径
    reshape_features(input_lmdb_path, output_lmdb_path)


260it [00:00, 1041.22it/s]

Reshaped features and saved data to /home/zjy/code/mycode/ocp/test_data/Cu_self/test_1


In [44]:
# 生成干扰数据

import numpy as np
import torch
import lmdb
import pickle
from torch_geometric.data import Data

def calculate_distances(pos, center_point):
    nearest_distances = np.linalg.norm(pos - center_point, axis=1)
    return torch.from_numpy(nearest_distances).float()

def generate_random_points_in_sphere(center, radius=1, num_points=10):
    center = np.array(center)  # 确保中心点为NumPy数组
    points = []
    for _ in range(num_points):
        # Generate a random point in a unit sphere
        u = np.random.uniform(0, 1)
        v = np.random.uniform(0, 1)
        theta = 2 * np.pi * u
        phi = np.arccos(2 * v - 1)
        r = radius * (np.random.uniform(0, 1) ** (1/3))
        x = r * np.sin(phi) * np.cos(theta)
        y = r * np.sin(phi) * np.sin(theta)
        z = r * np.cos(phi)
        random_point = np.array([x, y, z]) + center
        points.append(random_point)
    return points

def process_lmdb_for_distances(input_path, output_path):
    input_env = lmdb.open(input_path, readonly=True, map_size=1099511627776)
    output_env = lmdb.open(output_path, map_size=1099511627776)
    with input_env.begin() as input_txn, output_env.begin(write=True) as output_txn:
        cursor = input_txn.cursor()
        for key, value in cursor:
            data = pickle.loads(value)
            pos = data.pos.numpy()
            tags = data.tags.numpy()
            y = data.y.item()  # 转换为标量
            adsorbate_pos = pos[tags == 2]
            if len(adsorbate_pos) == 0:
                print("No adsorbate atoms found.")
                continue

            average_adsorbate_pos = adsorbate_pos.mean(axis=0)
            random_points = generate_random_points_in_sphere(average_adsorbate_pos)

            for i, point in enumerate(random_points):
                new_data = Data(**{k: v.clone() if torch.is_tensor(v) else v for k, v in data.items()})
                
                distances = calculate_distances(pos, point)
                new_data.distances = distances
                new_data.y = torch.tensor(y + np.random.uniform(-0.03, 0.03), dtype=torch.float64)
                new_data.fid = torch.tensor(i, dtype=torch.int64)  # Assign a unique fid for each new entry
                updated_value = pickle.dumps(new_data)
                output_txn.put(f"{key.decode('utf-8')}_copy_{i}".encode('utf-8'), updated_value)
    
    input_env.close()
    output_env.close()

# Set the paths directly in the script
input_path = "/home/zjy/code/mycode/ocp/test_data/all_ganrao/train+val_2"
output_path = "/home/zjy/code/mycode/ocp/test_data/all_ganrao/train+val_3"

process_lmdb_for_distances(input_path, output_path)


In [25]:
import lmdb
import pickle
import random

def shuffle_lmdb(input_lmdb_path, output_lmdb_path):
    # 打开旧的 LMDB 文件
    env_old = lmdb.open(input_lmdb_path, readonly=True, lock=False)
    
    # 打开新的 LMDB 文件
    env_new = lmdb.open(output_lmdb_path, map_size=int(1e12))
    
    try:
        with env_old.begin(write=False) as txn_old:
            cursor = txn_old.cursor()
            
            # 将所有键值对读取到内存中
            data_list = []
            for old_key, value in cursor:
                try:
                    data = pickle.loads(value)
                    data_list.append(data)
                except Exception as e:
                    print(f"Error processing key {old_key.decode()}: {e}")
            
            # 打乱数据顺序
            random.shuffle(data_list)
        
        # 确保新的环境未关闭
        if not env_new:
            raise ValueError("New LMDB environment is closed or invalid.")
        
        # 将打乱后的数据写入新的 LMDB 文件，并重新赋予 key 值
        with env_new.begin(write=True) as txn_new:
            for new_key_index, data in enumerate(data_list):
                new_key = f"{new_key_index}".encode("ascii")
                txn_new.put(new_key, pickle.dumps(data, protocol=-1))
            
            txn_new.commit()
            env_new.sync()
    finally:
        env_old.close()
        env_new.close()

# 使用示例
input_lmdb_path = '/home/zjy/code/mycode/ocp/test_data/H/train'
output_lmdb_path = '/home/zjy/code/mycode/ocp/test_data/H/train_gai'
shuffle_lmdb(input_lmdb_path, output_lmdb_path)


Error: Attempt to operate on closed/deleted/dropped object.